# Chapter 9: Retrieval — Putting It to Work
## Topic 9: Contextual Retrieval (Anthropic's Technique)

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This exists

- Every chunking discussion so far (Chapter 5's ingestion pipeline, referenced throughout Chapter 7) has implicitly accepted a real cost of chunking: once a document is split into pieces, each individual chunk loses the surrounding context that made it fully meaningful in the original document. A chunk that says "the penalty is 1 percent" has lost the information about *which specific product* or *which specific scenario* that sentence was originally answering, if that detail lived in a different paragraph.
- Contextual Retrieval is Anthropic's published technique for addressing this directly: before a chunk is embedded and indexed (at ingestion time, not at query time), prepend a short, LLM-generated explanation of what this chunk is about *in the context of the whole document it came from* — then embed and index this contextualized version instead of the bare original chunk.
- The core mechanism, stated precisely: for every chunk, make one call to a fast, cheap model, passing it the *full original document* and the *specific chunk*, and ask it to write a short (typically one or two sentence) explanatory prefix situating that chunk within the document. This prefix gets prepended to the chunk before embedding and before being added to the BM25 index — the chunk stored and searched is the *contextualized* version, not the original bare text.
- Why this specifically matters for this project: Chapter 7 Topic 6 (MMR) already established that this project's knowledge base has real structural redundancy — the same fact restated across an FAQ, a policy document, and an SOP, each in a different style. A bare chunk like "Step 3: apply the 1 percent penalty deduction" (from the SOP) loses the information that this step specifically belongs to the *premature withdrawal* procedure once separated from its surrounding steps — contextual retrieval's prefix would restore exactly this missing situating information.


### 2. Internal Working — Step by Step

**The contextual retrieval pipeline, step by step:**

1. **Document ingestion proceeds as before** (Chapter 5) — a source document is loaded and split into chunks using whatever chunking strategy is in use (Topic 10 covers evaluating this choice).
2. **For each chunk, make one contextualization call** to a fast, cheap model: the prompt includes the *entire original document* (or a representative portion of it) plus the *specific chunk* being processed, and asks for a short, situating explanation — something like "This chunk is from the FD premature withdrawal SOP, specifically the step describing penalty calculation."
3. **Prepend this generated context to the chunk**, producing a new, contextualized version of the chunk — this is the *only* version that gets embedded and indexed going forward; the bare original chunk is not separately indexed.
4. **Index the contextualized chunk in both the BM25 index and the dense/vector index** — contextual retrieval is explicitly designed to improve both retrieval methods in Chapter 7's hybrid pipeline, not just one.
5. **Retrieval proceeds exactly as built in Chapter 7** — hybrid BM25+dense+RRF, reranking, MMR — completely unchanged. The only thing that's different is what's actually stored and searchable: contextualized chunks instead of bare ones.

**Why this improves both BM25 and dense retrieval, for genuinely different reasons:**

- **For BM25 specifically:** the contextualizing prefix adds real, searchable vocabulary the bare chunk didn't have — a chunk that never says the word "premature" on its own now has that word in its prefix if the contextualization correctly identifies that's what the chunk relates to, directly increasing the chance of a genuine token match for queries using that word.
- **For dense retrieval specifically:** the contextualizing prefix gives the embedding model more complete, self-contained semantic content to encode — a bare, decontextualized sentence produces a vector representing only that isolated sentence's meaning; a contextualized version produces a vector that better represents what the chunk is actually *about* within its source document, which Chapter 7 Topic 2 established matters given this project's already-thin discrimination gap for short, ambiguous text.

**The cost this technique explicitly trades for that improvement:** one additional model call *per chunk*, at ingestion time — this is a one-time cost (paid once when a document is ingested or re-ingested, not on every query), which is the crucial detail that makes the trade-off favorable despite sounding expensive at first glance.


### 3. How This Is Implemented in This Project

- Contextual retrieval slots into the ingestion pipeline (Chapter 5) as an additional processing step between chunking and indexing — for each chunk produced by the existing chunking strategy, one contextualization call runs before that chunk (now contextualized) gets embedded and added to both the BM25 and dense indexes built in Chapter 7.
- This is a particularly good fit for exactly the redundancy pattern Chapter 7 Topic 6 (MMR) identified: the SOP's numbered steps, which lose their surrounding procedural context when chunked individually (Chapter 7 Topic 1's own example — "Step 3: apply the 1 percent penalty deduction" — is precisely the kind of chunk that benefits most from an explicit "this step belongs to the premature withdrawal procedure" prefix).
- Since this changes what's actually stored and indexed, adopting this technique means re-ingesting the existing knowledge base with the new contextualization step added — the existing chunking strategy, the existing BM25 index construction, and the existing embedding pipeline all stay structurally the same; only the *content* being fed into them changes.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Cost is a one-time ingestion cost, not a per-query cost — this is the central economic argument for the technique, and needs to be understood precisely.** One additional model call per chunk at ingestion time, for a knowledge base with a modest number of chunks (this project's small, ~17-page knowledge base), is a small, one-time cost. This is fundamentally different from, and much cheaper than, any technique (like HyDE, Chapter 8 Topic 7) that adds a model call on every single query at production volume — contextual retrieval trades a bounded, one-time ingestion cost for an ongoing, unbounded query-time quality improvement.
- **Re-ingestion cost scales with how often source documents change.** If the underlying policy documents update frequently, contextualization needs to re-run for affected chunks each time — for content that changes rarely (which matches this project's actual policy document update pattern, discussed in Chapter 7 Topic 1's IDF-stability concerns), this is a manageable, infrequent cost; for a genuinely fast-changing knowledge base, this recurring re-ingestion cost needs to be weighed more carefully.
- **The contextualization call's own reliability matters:** a poorly-contextualized chunk (one where the generated prefix mischaracterizes what the chunk is actually about) could in principle make retrieval *worse* rather than better for that specific chunk — this needs the same evidence-based validation as every other technique in this notebook series: measure retrieval quality (Chapter 7 Topic 9's Recall@K/MRR/NDCG@K) before and after adopting contextual retrieval on the same evaluation set, rather than assuming the technique helps by reputation alone.
- **Monitoring:** track retrieval evaluation metrics specifically for the redundancy-prone content types (SOP steps, in this project's case) before and after contextualization, since this is where the theory predicts the largest measurable improvement — a technique that helps broadly but doesn't specifically improve the case it's most theoretically suited for is a signal worth investigating.
- **Latency:** contextual retrieval adds zero query-time latency — the contextualization work happens entirely at ingestion time, before any query ever arrives, which is a meaningful practical advantage over query-time techniques like HyDE that add latency to every single request.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Contextual retrieval vs simply improving the chunking strategy itself (Topic 10):** these are complementary, not competing — a better chunking strategy (respecting natural document boundaries, appropriate chunk sizes) reduces how much context gets lost in the first place, while contextual retrieval restores context that's lost even under a well-designed chunking strategy, since some cross-chunk context (like "this step belongs to procedure X") can't be fully solved by chunk boundaries alone. Both should be considered together, not treated as alternative fixes for the same problem.
- **Whether to use a fast, cheap model for contextualization vs a more capable one:** since this cost is paid once per chunk at ingestion time rather than per query, the model choice trade-off here is different from a query-time technique — a more capable model could produce meaningfully better contextualization at a still-modest total ingestion cost (since it's paid once, not per query), making this a case where investing in higher quality is more easily justified than it would be for a per-query cost.
- **The real trade-off this project specifically faces:** given the knowledge base's small, modest chunk count, the total ingestion cost of contextualizing every chunk is genuinely small in absolute terms — the more relevant design question is whether the measured retrieval-quality improvement (validated via Chapter 7 Topic 9's evaluation harness) is large enough to justify the added ingestion-pipeline complexity, not whether the cost itself is prohibitive.


### 6. Alternatives and When to Use Each

- **Contextual retrieval (this topic's technique):** the right choice when redundancy and cross-chunk context loss are measured, real problems — exactly this project's situation, given Chapter 7 Topic 6's already-documented redundancy pattern across FAQ/policy/SOP restatements of the same facts.
- **Better chunking strategy alone (Topic 10):** a complementary, not competing, first step — worth optimizing before or alongside contextual retrieval, since a poor chunking strategy can undermine even a well-implemented contextualization step.
- **HyDE (Chapter 8 Topic 7):** a different technique solving a related but distinct problem — HyDE addresses query-side phrasing and vocabulary mismatch at query time; contextual retrieval addresses document-side context loss at ingestion time. These are not mutually exclusive and can be combined.
- **No contextualization, relying entirely on reranking and MMR (Chapter 7 Topics 6-7) to compensate for context loss at retrieval time:** a reasonable baseline if contextualization's added ingestion complexity isn't measured to be worth it — but reranking and MMR operate on whatever chunks were retrieved; they can't recover meaning that was never captured in the chunk's stored representation in the first place, which is specifically what contextual retrieval addresses.


### 7. Common Mistakes and Production Failures

- Confusing contextual retrieval's one-time ingestion cost with an ongoing per-query cost, and rejecting the technique based on a cost concern that doesn't actually apply to how it's structured.
- Adopting contextual retrieval without measuring its actual effect on retrieval evaluation metrics (Chapter 7 Topic 9) before and after, assuming it helps purely based on its published reputation rather than validating it against this project's own specific data.
- Applying contextualization inconsistently — contextualizing chunks in the dense index but not the BM25 index, or vice versa — losing the technique's intended benefit for whichever retrieval method didn't receive the contextualized version.
- Treating contextual retrieval as a substitute for good chunking strategy, rather than a complementary technique — a poorly-chunked document produces poor chunks that contextualization can improve somewhat but not fully compensate for.
- Not accounting for re-ingestion cost when source documents update, if the technique is adopted for a knowledge base with more frequently-changing content than this project's currently has.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What problem does contextual retrieval solve, and when does its cost get paid?
  A: It solves the problem of chunks losing meaningful surrounding context once a document is split apart — a chunk in isolation may lack information about what broader topic or procedure it belongs to. Its cost is paid once per chunk, at ingestion time, when a document is first processed — not on every query, which is what makes it economically different from query-time techniques like HyDE.

- Q: Why does contextual retrieval improve both BM25 and dense retrieval, and for different reasons?
  A: For BM25, the added contextualizing prefix introduces real, searchable vocabulary the bare chunk lacked, directly increasing the chance of genuine token overlap with a relevant query. For dense retrieval, the prefix gives the embedding model more complete, self-contained semantic content to encode, producing a vector that better represents the chunk's actual meaning within its source document rather than just the isolated sentence's meaning.

**Intermediate**

- Q: This project's knowledge base has documented redundancy — the same fact restated across an FAQ, a policy document, and an SOP (established in Chapter 7 Topic 6's MMR discussion). How does contextual retrieval specifically help with the SOP case?
  A: An SOP's individual numbered steps, chunked in isolation, lose the information about which broader procedure they belong to — a step saying "apply the 1 percent penalty deduction" doesn't say on its own that this belongs to the premature withdrawal procedure specifically. Contextual retrieval's generated prefix would explicitly restore this situating information, making the chunk searchable and interpretable on its own even after being separated from its surrounding steps.

- Q: How does contextual retrieval's cost profile compare to HyDE's?
  A: Contextual retrieval's cost is paid once per chunk at ingestion time — a bounded, one-time cost. HyDE's cost is paid on every single query at retrieval time — an ongoing, unbounded cost that scales with production query volume. This makes contextual retrieval's total cost more predictable and, for a knowledge base that doesn't change extremely frequently, considerably cheaper in aggregate at meaningful query volume.

**Advanced**

- Q: How would you decide whether contextual retrieval is actually worth adopting for a specific project?
  A: Measure retrieval evaluation metrics (Recall@K, MRR, NDCG@K from Chapter 7 Topic 9) on the same evaluation set, before and after re-ingesting the knowledge base with contextualization applied — specifically checking whether the improvement concentrates in exactly the content types theoretically most likely to benefit (redundant, structurally-fragmented content like SOP steps). A technique that shows broad but unfocused improvement, or no measurable improvement at all despite the theoretical argument, is a signal to investigate before committing to the added ingestion-pipeline complexity permanently.

- Q: A teammate argues that better chunking strategy alone (Topic 10) makes contextual retrieval unnecessary. How do you respond?
  A: Better chunking reduces how much context is lost by choosing better boundaries, but no chunking strategy can fully prevent all cross-chunk context loss — some information (like "this step belongs to this specific procedure") inherently exists at a level above any individual chunk's boundaries, however well-chosen those boundaries are. The two techniques are complementary: better chunking is a genuinely worthwhile investment regardless, and contextual retrieval addresses a category of context loss that chunking strategy alone structurally cannot fully solve.

**Scenario-based**

- Q: After implementing contextual retrieval, you notice one specific chunk's generated contextualization is subtly wrong — it mischaracterizes what the chunk is about. What's your response, and how would you catch this systematically rather than by chance?
  A: For the immediate case, flag and manually correct that specific chunk's contextualization, then re-index it. Systematically, this points to the need for a spot-check or sampling process on generated contextualizations, similar to how any other model-generated content in this pipeline (like citations, Chapter 8 Topic 2) benefits from periodic human review rather than being trusted unconditionally — and if retrieval evaluation metrics for that specific document or content type show a regression rather than an improvement after contextualization, that's a measurable signal pointing at exactly this kind of contextualization-quality issue.

**Follow-up questions to expect:**

- "How would you decide which model to use for the contextualization step, given it's a one-time cost per chunk?"
- "What would you monitor to detect if contextualization quality degrades over time, for example after a knowledge base restructuring?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Contextual retrieval is a specific application of a general principle: enrich content with metadata or explanatory context BEFORE indexing, rather than trying to compensate for missing context entirely at query time.** This "fix it upstream, at ingestion, rather than downstream, at query time" philosophy echoes Chapter 7 Topic 8's metadata filtering (attaching structured metadata at ingestion time to enable better search-time filtering) — both techniques improve retrieval by enriching what gets indexed, rather than by making the query-time search logic itself more sophisticated.
- **The one-time-cost vs per-query-cost distinction is a broadly reusable framework for evaluating any proposed technique in a retrieval pipeline:** whenever a new technique is proposed, asking "does this cost get paid once, at ingestion, or does it get paid on every query" is often the single most important question for understanding its true cost profile at production scale — this framework applies well beyond this specific technique.
- **This technique connects directly back to Chapter 7 Topic 3's broader theme (DPR, ColBERT, SPLADE) of using more sophisticated processing to produce better representations for search:** contextual retrieval is a comparatively simple, LLM-based way to produce a better representation of a chunk before embedding it, conceptually related to (though much simpler than) those more architecturally sophisticated representation-learning techniques.

### 10. Quick Revision Summary (for last-minute interview prep)

> Contextual retrieval addresses the real cost of chunking: individual chunks lose the surrounding context that made them fully meaningful in their original document. The technique adds one model call per chunk at ingestion time — before embedding and indexing — asking a fast, cheap model to generate a short, situating explanation of what the chunk is about within its source document, then prepends this to the chunk before it's added to both the BM25 and dense indexes. This improves BM25 by adding genuinely new, searchable vocabulary, and improves dense retrieval by giving the embedding model more complete, self-contained semantic content to encode. Its defining economic characteristic is that its cost is paid once, at ingestion, not on every query — fundamentally different from and typically cheaper than query-time techniques like HyDE at meaningful production volume. This project's own already-documented content redundancy (Chapter 7 Topic 6's finding that the same fact gets restated across FAQ, policy, and SOP documents, with SOP steps specifically losing their procedural context when chunked individually) makes this a particularly well-matched technique to validate — but as with every technique in this notebook series, its actual benefit should be measured against the retrieval evaluation harness (Chapter 7 Topic 9) rather than assumed from reputation alone.


### Module 1: Setup — Bare vs Contextualized Chunks

Build the exact kind of fragmented SOP-step chunk that loses procedural context when isolated, and simulate what a real contextualization call would prepend to it.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Bare vs contextualized chunks
#
# We cannot call a real LLM offline to generate contextualization.
# We SIMULATE the contextualization step -- but the chunk content and
# the retrieval demonstration in Modules 2-3 are REAL and testable.
# The simulated prefix below is written the way a real contextualization
# call would realistically produce it: short, situating, factual.
# ------------------------------------------------------------------

# the FULL original SOP document this chunk comes from -- needed by a
# real contextualization call to understand what the isolated chunk means
FULL_SOP_DOCUMENT = """
FD Premature Withdrawal Standard Operating Procedure

Step 1: Verify FD account for premature withdrawal eligibility.
Step 2: Check FD premature withdrawal penalty applicable rate.
Step 3: Apply the 1 percent penalty deduction on the FD interest rate.
Step 4: Process the withdrawal and credit the remaining amount.
Step 5: Generate a receipt for the customer confirming penalty deduction.
"""

# the BARE chunk -- what a chunking strategy alone would isolate, with
# ALL its surrounding procedural context stripped away
BARE_CHUNK = "Step 3: Apply the 1 percent penalty deduction on the FD interest rate."

def simulate_contextualization(chunk: str, full_document: str) -> str:
    """SIMULATES what a real contextualization model call would produce,
    given the full document and this specific chunk. A real
    implementation calls client.messages.create() with both as input;
    we hand-write a realistic, situating prefix here."""
    return ("This is Step 3 of the FD premature withdrawal procedure, describing "
            "how the penalty is calculated. ")

contextualizing_prefix = simulate_contextualization(BARE_CHUNK, FULL_SOP_DOCUMENT)
CONTEXTUALIZED_CHUNK = contextualizing_prefix + BARE_CHUNK

print("=" * 70)
print("MODULE 1: BARE vs CONTEXTUALIZED CHUNK")
print("=" * 70)
print(f"BARE chunk (what chunking alone produces):")
print(f"  '{BARE_CHUNK}'")
print(f"\nCONTEXTUALIZED chunk (bare chunk + generated situating prefix):")
print(f"  '{CONTEXTUALIZED_CHUNK}'")

print("\nNotice the bare chunk never says 'premature withdrawal' or")
print("'procedure' at all -- only 'penalty' and 'interest rate' survive")
print("from the isolated step. The contextualized version restores the")
print("missing situating vocabulary a query would realistically use.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: BARE vs CONTEXTUALIZED CHUNK
BARE chunk (what chunking alone produces):
  'Step 3: Apply the 1 percent penalty deduction on the FD interest rate.'

CONTEXTUALIZED chunk (bare chunk + generated situating prefix):
  'This is Step 3 of the FD premature withdrawal procedure, describing how the penalty is calculated. Step 3: Apply the 1 percent penalty deduction on the FD interest rate.'

Notice the bare chunk never says 'premature withdrawal' or
'procedure' at all -- only 'penalty' and 'interest rate' survive
from the isolated step. The contextualized version restores the
missing situating vocabulary a query would realistically use.

Module 1 complete. Run Module 2 next.


### Module 2: BM25 Retrieval — Bare vs Contextualized, Real Vocabulary Match

Build two real BM25 indexes (one with bare chunks, one with contextualized chunks) and measure the actual difference for a realistic customer query.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: BM25 retrieval quality -- bare vs contextualized indexes
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

# a small knowledge base -- the target SOP step, plus distractor chunks
# from OTHER, unrelated procedures/documents
BARE_KNOWLEDGE_BASE = [
    BARE_CHUNK,
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "Nomination facility is available for all Fixed Deposit account holders.",
]

CONTEXTUALIZED_KNOWLEDGE_BASE = [
    CONTEXTUALIZED_CHUNK,
    "This is from the FD product guide, describing senior citizen benefits. Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "This is from the FD product guide, describing nomination options. Nomination facility is available for all Fixed Deposit account holders.",
]

# a realistic customer query about the PREMATURE WITHDRAWAL PROCEDURE
# specifically -- using the word "procedure" and "premature", which
# only the CONTEXTUALIZED chunk actually contains
QUERY = "what is the procedure for premature withdrawal penalty calculation"

bare_bm25 = BM25Okapi([tokenize(c) for c in BARE_KNOWLEDGE_BASE])
contextualized_bm25 = BM25Okapi([tokenize(c) for c in CONTEXTUALIZED_KNOWLEDGE_BASE])

bare_scores = bare_bm25.get_scores(tokenize(QUERY))
contextualized_scores = contextualized_bm25.get_scores(tokenize(QUERY))

print("=" * 70)
print("BM25 RETRIEVAL: BARE INDEX vs CONTEXTUALIZED INDEX")
print("=" * 70)
print(f"Query: '{QUERY}'\n")

print("BARE index scores:")
for text, score in zip(BARE_KNOWLEDGE_BASE, bare_scores):
    print(f"  score={score:.4f} | {text[:60]}...")

print("\nCONTEXTUALIZED index scores:")
for text, score in zip(CONTEXTUALIZED_KNOWLEDGE_BASE, contextualized_scores):
    print(f"  score={score:.4f} | {text[:60]}...")

bare_target_score = bare_scores[0]
contextualized_target_score = contextualized_scores[0]

print(f"\nTarget chunk's score in BARE index:           {bare_target_score:.4f}")
print(f"Target chunk's score in CONTEXTUALIZED index:  {contextualized_target_score:.4f}")

if contextualized_target_score > bare_target_score:
    improvement = contextualized_target_score - bare_target_score
    print(f"\nCONFIRMED: contextualization improved the target chunk's BM25")
    print(f"score by {improvement:.4f} for this query, by adding genuinely new,")
    print(f"searchable vocabulary ('procedure', 'premature', 'withdrawal') that")
    print(f"the bare chunk structurally never contained on its own.")

print("\nModule 2 complete. Run Module 3 next.")


BM25 RETRIEVAL: BARE INDEX vs CONTEXTUALIZED INDEX
Query: 'what is the procedure for premature withdrawal penalty calculation'

BARE index scores:
  score=1.2031 | Step 3: Apply the 1 percent penalty deduction on the FD inte...
  score=0.0000 | Senior citizens receive an additional 0.5 percent interest o...
  score=1.1045 | Nomination facility is available for all Fixed Deposit accou...

CONTEXTUALIZED index scores:
  score=1.6454 | This is Step 3 of the FD premature withdrawal procedure, des...
  score=0.0114 | This is from the FD product guide, describing senior citizen...
  score=0.5701 | This is from the FD product guide, describing nomination opt...

Target chunk's score in BARE index:           1.2031
Target chunk's score in CONTEXTUALIZED index:  1.6454

CONFIRMED: contextualization improved the target chunk's BM25
score by 0.4423 for this query, by adding genuinely new,
searchable vocabulary ('procedure', 'premature', 'withdrawal') that
the bare chunk structurally never contain

### Module 3: Dense Retrieval — Bare vs Contextualized, Real Semantic Comparison

Same comparison using offline TF-IDF+SVD dense vectors (Chapter 7 Topic 2's approach), showing the semantic-side improvement.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Dense retrieval -- bare vs contextualized, real vectors
# ------------------------------------------------------------------

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def normalize_vector(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine_similarity(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0

def build_dense_vectors(texts, query, n_components=3):
    all_texts = texts + [query]
    vectorizer = TfidfVectorizer()
    sparse = vectorizer.fit_transform(all_texts)
    n_comp = min(n_components, sparse.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    dense = svd.fit_transform(sparse)
    dense_norm = np.array([normalize_vector(v) for v in dense])
    return dense_norm[:-1], dense_norm[-1]  # doc vectors, query vector


bare_doc_vecs, bare_query_vec = build_dense_vectors(BARE_KNOWLEDGE_BASE, QUERY)
contextualized_doc_vecs, contextualized_query_vec = build_dense_vectors(CONTEXTUALIZED_KNOWLEDGE_BASE, QUERY)

bare_dense_scores = [cosine_similarity(bare_query_vec, v) for v in bare_doc_vecs]
contextualized_dense_scores = [cosine_similarity(contextualized_query_vec, v) for v in contextualized_doc_vecs]

print("=" * 70)
print("DENSE RETRIEVAL: BARE vs CONTEXTUALIZED (offline TF-IDF+SVD vectors)")
print("=" * 70)
print(f"Query: '{QUERY}'\n")

print(f"BARE index -- target chunk dense similarity:           {bare_dense_scores[0]:.4f}")
print(f"CONTEXTUALIZED index -- target chunk dense similarity: {contextualized_dense_scores[0]:.4f}")

if contextualized_dense_scores[0] > bare_dense_scores[0]:
    print(f"\nCONFIRMED: contextualization also improved the DENSE retrieval")
    print(f"score, not just BM25 -- the added prefix gives the embedding more")
    print(f"complete, self-contained semantic content to encode, exactly the")
    print(f"dual improvement (BM25 AND dense) the theory describes.")

print("\n" + "=" * 70)
print("SUMMARY: BOTH RETRIEVAL METHODS, BOTH IMPROVED")
print("=" * 70)
col_method, col_bare, col_ctx, col_imp = "Method", "Bare score", "Contextualized score", "Improved?"
print(f"{col_method:<12} | {col_bare:>12} | {col_ctx:>21} | {col_imp:>10}")
print("-" * 65)
bm25_improved = str(contextualized_target_score > bare_target_score)
dense_improved = str(contextualized_dense_scores[0] > bare_dense_scores[0])
print(f"{'BM25':<12} | {bare_target_score:>12.4f} | {contextualized_target_score:>21.4f} | {bm25_improved:>10}")
print(f"{'Dense':<12} | {bare_dense_scores[0]:>12.4f} | {contextualized_dense_scores[0]:>21.4f} | {dense_improved:>10}")

print("\nModule 3 complete. All Chapter 9 Topic 9 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 9 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Contextual retrieval prepends an LLM-generated, situating explanation
  to each chunk BEFORE embedding and indexing -- at INGESTION time,
  paid ONCE per chunk, never on a per-query basis.

  Improves BOTH BM25 (new searchable vocabulary) AND dense retrieval
  (more complete semantic content to encode) -- demonstrated concretely
  with real scores for both methods in this notebook.

  Best-matched to exactly this project's already-documented redundancy
  pattern (Chapter 7 Topic 6) -- SOP steps losing procedural context
  when chunked in isolation.

  One-time ingestion cost vs HyDE's per-query cost is the key economic
  distinction -- always ask "paid once or paid every query" for any
  new retrieval technique under consideration.

  Complementary to, not a substitute for, good chunking strategy
  (Topic 10) -- both should be considered together.
""")


DENSE RETRIEVAL: BARE vs CONTEXTUALIZED (offline TF-IDF+SVD vectors)
Query: 'what is the procedure for premature withdrawal penalty calculation'

BARE index -- target chunk dense similarity:           0.4395
CONTEXTUALIZED index -- target chunk dense similarity: 0.7543

CONFIRMED: contextualization also improved the DENSE retrieval
score, not just BM25 -- the added prefix gives the embedding more
complete, self-contained semantic content to encode, exactly the
dual improvement (BM25 AND dense) the theory describes.

SUMMARY: BOTH RETRIEVAL METHODS, BOTH IMPROVED
Method       |   Bare score |  Contextualized score |  Improved?
-----------------------------------------------------------------
BM25         |       1.2031 |                1.6454 |       True
Dense        |       0.4395 |                0.7543 |       True

Module 3 complete. All Chapter 9 Topic 9 modules done.

CHAPTER 9 TOPIC 9 -- KEY POINTS TO REMEMBER

  Contextual retrieval prepends an LLM-generated, situating explanat